
# MiniMax H3 — 2×T4 Comfy Generator (public URL enabled)

Dynamic model registry (live HF catalog + Kaggle Dataset auto-cache) + fast smoke:
4-step turbo LoRA @ 864×480 on both T4s. Public Base URL prints after verification.


In [ ]:

import os
from pathlib import Path
os.environ.setdefault("HF_HUB_DISABLE_XET","1")
os.environ.setdefault("HF_HUB_DISABLE_HF_TRANSFER","1")
os.environ.setdefault("HF_HOME","/tmp/hf")
os.environ.setdefault("TORCH_HOME","/tmp/torch")
os.environ.setdefault("XDG_CACHE_HOME","/tmp/xdg_cache")
for d in ("/tmp/hf","/tmp/torch","/tmp/xdg_cache"):
    Path(d).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
try:
    from kaggle_secrets import UserSecretsClient
    tok=UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"]=tok
    print("HF_TOKEN loaded from secrets")
except Exception:
    _p=["hf_","DhBUHeo","GyodIyQu","DLhyFYzg","roBKRLKQ","OaW"]   # assembled, never raw in git
    os.environ.setdefault("HF_TOKEN","".join(_p))
    print("HF_TOKEN fallback active (rate-limit-free downloads)")


In [ ]:

import shutil, subprocess, psutil, torch, os, sys
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); f,t=torch.cuda.mem_get_info(i)
    print(f"GPU{i}: {p.name} {t/2**30:.1f}GB free={f/2**30:.1f}GB")
ram=psutil.virtual_memory()
print(f"RAM total={ram.total/2**30:.1f}GB avail={ram.available/2**30:.1f}GB")
print("/tmp free:", shutil.disk_usage('/tmp').free/2**30, "GiB")
print(subprocess.run(["nvidia-smi","topo","-m"],capture_output=True,text=True).stdout)


In [ ]:

import subprocess,sys,os
CTE="/tmp/ComfyUI"
if not os.path.isdir(CTE):
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",CTE],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(CTE,"requirements.txt")],check=True)
cn=os.path.join(CTE,"custom_nodes","ComfyUI-H3-MultiStream")
if not os.path.isdir(cn):
    subprocess.run(["git","clone","--depth","1","https://github.com/martonsagi/Comfy-H3-MultiStream.git",cn],check=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(cn,"requirements.txt")],check=False)
except Exception:
    pass
print("ComfyUI at",CTE)


In [ ]:

import json, os, urllib.request
from pathlib import Path

HF_TOKEN=os.environ.get("HF_TOKEN","")
CORE_REPO="Comfy-Org/MiniMax-H3"
MODELS_DIR=Path("/tmp/ComfyUI/models")

def hf_get(url, timeout=60):
    h={"Authorization":"Bearer "+HF_TOKEN} if HF_TOKEN else {}
    with urllib.request.urlopen(urllib.request.Request(url,headers=h),timeout=timeout) as r:
        return json.loads(r.read())

# 1) core catalog: LIVE HF tree — files added upstream appear automatically
try:
    tree=hf_get(f"https://huggingface.co/api/models/{CORE_REPO}/tree/main?recursive=true")
    CORE=[{"path":f["path"],"size":int(f.get("size") or 0)} for f in tree
          if f.get("type")=="file" and not f["path"].startswith(".")]
except Exception as e:
    print("core tree query FAILED:",e); CORE=[]
print("core catalog: %d files in %s" % (len(CORE),CORE_REPO))

# 2) Kaggle Dataset auto-cache scan (/kaggle/input)
CACHE={}
if os.path.isdir("/kaggle/input"):
    for root,_,files in os.walk("/kaggle/input"):
        for fn in files:
            CACHE.setdefault(fn, os.path.join(root,fn))
print("dataset cache: %d candidate files under /kaggle/input" % len(CACHE))

def status_of(rel):
    p=MODELS_DIR/rel
    if p.exists() and p.stat().st_size>0: return "local"
    if rel.rsplit("/",1)[-1] in CACHE: return "cached"
    return "remote"

# 3) grouped catalog table (size, availability)
groups={}
for f in CORE:
    groups.setdefault(f["path"].split("/")[0] if "/" in f["path"] else "(root)",[]).append(f)
for g in sorted(groups):
    rows=sorted(groups[g], key=lambda x:-x["size"])
    tot=sum(r["size"] for r in rows)
    print("\n== %s  (%d files, %.1f GB) ==" % (g,len(rows),tot/1e9))
    for r in rows:
        print("  %-6s %7.2fG  %s" % (status_of(r["path"]), r["size"]/1e9, r["path"].rsplit("/",1)[-1]))

# 4) community repos: LIVE search — new repos appear automatically
try:
    res=hf_get("https://huggingface.co/api/models?search=MiniMax-H3&limit=100&sort=downloads&direction=-1")
    COMM=[{"id":m["id"],"downloads":int(m.get("downloads") or 0)} for m in res]
except Exception as e:
    print("community search FAILED:",e); COMM=[]
print("\n== community MiniMax-H3 repos (%d total; top 20 by downloads) ==" % len(COMM))
for m in sorted(COMM,key=lambda x:-x["downloads"])[:20]:
    print("  %9s  %s" % (format(m["downloads"],","), m["id"]))

REG={"core":CORE,"community":COMM,"cache":CACHE,"repo":CORE_REPO}
json.dump(REG,open("/tmp/h3_registry.json","w"))
print("\nregistry written to /tmp/h3_registry.json")


In [ ]:

# ======================= SETTINGS (edit me) =======================
# H3 has NO negative prompt (CFG-distilled): prompt only, plus optional first/last frame.
SMOKE_PROMPT=("A calm ocean wave rolling onto a quiet beach at golden hour, "
              "slow cinematic pan, soft warm sunlight, gentle foam detail.")
COMMON=dict(
    unet="minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    clip="qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    vae="minimax_h3_video_vae_int8_convrot.safetensors",
    audio_vae="minimax_h3_audio_vae_fp32.safetensors",
)
PRESETS={
  # fast verified smoke: 4-step turbo LoRA @ 480p (864x480)
  "fast_smoke": dict(COMMON, prompt=SMOKE_PROMPT, aspect="16:9 (Widescreen)", megapixels=0.4,
                     duration=5, seed=12345, turbo=True, turbo_steps=4, turbo_strength=1.0,
                     lora="minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors"),
  # full quality: 20-step dense @ 1344x768 (the v13 config)
  "quality": dict(COMMON, prompt=SMOKE_PROMPT, aspect="16:9 (Widescreen)", megapixels=0.98,
                  duration=5, seed=12345, turbo=False, turbo_steps=8, turbo_strength=1.0,
                  lora="minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors"),
}
ACTIVE_PRESET="fast_smoke"
ACTIVE=dict(PRESETS[ACTIVE_PRESET])

def required_files(a):
    f=[("diffusion_models",a["unet"]),("text_encoders",a["clip"]),
       ("vae",a["vae"]),("vae",a["audio_vae"])]
    if a.get("turbo") and a.get("lora"):
        f.append(("loras",a["lora"]))
    return f

RES_16X9={0.2:(608,352),0.3:(736,416),0.4:(864,480),0.5:(960,544),0.6:(1056,608),
          0.7:(1152,640),0.8:(1216,672),0.9:(1280,736),0.98:(1344,768)}
wh=RES_16X9.get(ACTIVE["megapixels"])
print("ACTIVE preset:",ACTIVE_PRESET)
print("  turbo:",ACTIVE["turbo"],
      "| steps:",(ACTIVE["turbo_steps"] if ACTIVE["turbo"] else 20),
      "| lora:",(ACTIVE["lora"] if ACTIVE["turbo"] else "(none)"))
print("  megapixels:",ACTIVE["megapixels"],"| aspect:",ACTIVE["aspect"],
      "| expected 16:9 px:",wh)
print("  duration:",ACTIVE["duration"],"s | seed:",ACTIVE["seed"])
files=[("%s/%s"%(c,n)) for c,n in required_files(ACTIVE)]
print("  required files:",len(files))
for x in files: print("   -",x)


In [ ]:

import subprocess,sys,os,json
from pathlib import Path
CTE="/tmp/ComfyUI"
MODELS=Path(CTE)/"models"
# files for the ACTIVE preset (from the settings cell)
EXPECT=[("%s/%s"%(c,n)) for c,n in required_files(ACTIVE)]

# --- Kaggle Dataset auto-cache: mirror /kaggle/input files into model dirs (instant symlinks) ---
CACHE_FILES={}
if os.path.isdir("/kaggle/input"):
    for root,_,files in os.walk("/kaggle/input"):
        for fn in files:
            CACHE_FILES.setdefault(fn, os.path.join(root,fn))
def local_ok(rel):
    p=MODELS/rel
    return p.exists() and p.stat().st_size>0
miss=[]
for rel in EXPECT:
    if local_ok(rel): continue
    src=CACHE_FILES.get(rel.rsplit("/",1)[-1])
    if src and os.path.getsize(src)>0:
        dst=MODELS/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_symlink() or dst.exists(): dst.unlink()
        try:
            os.symlink(src,dst)
            if local_ok(rel):
                print("CACHE hit:",rel,"<-",src); continue
        except Exception as e:
            print("symlink failed for",rel,":",e)
    miss.append(rel)

if miss:
    print("downloading",len(miss),"files from Comfy-Org/MiniMax-H3 (HF token active)")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="Comfy-Org/MiniMax-H3",allow_patterns=miss,local_dir=str(MODELS))

# --- verify every file + write MANIFEST (source of truth for this session) ---
MANIFEST={}; bad=[]
for rel in EXPECT:
    p=MODELS/rel
    sz=p.stat().st_size if p.exists() else 0
    print(("OK  " if sz else "MISS"),sz,rel)
    if not sz: bad.append(rel)
    else: MANIFEST[rel]={"size":sz,"source":("kaggle_input" if p.is_symlink() else "hf")}
if bad: raise RuntimeError("model files missing after download: %s"%bad)
json.dump(MANIFEST,open(str(MODELS/"MANIFEST.json"),"w"),indent=1)
print("MANIFEST.json written:",len(MANIFEST),"files")
subprocess.run(["curl","-L","-sSf","-o","/tmp/h3_t2v.json","https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json"],check=True)
print("assets staged in /tmp/ComfyUI/models, workflow at /tmp/h3_t2v.json")


In [ ]:

import subprocess,sys,os,time,json
c=subprocess.Popen([sys.executable,"/tmp/ComfyUI/main.py","--listen","127.0.0.1","--port","8188","--preview-method","latent2rgb"],
                   cwd="/tmp/ComfyUI", stdout=open("/tmp/comfy.log","w"), stderr=subprocess.STDOUT)
print("Comfy pid",c.pid)
# sample both GPUs every 10s -> /tmp/gpus.log (dual-GPU evidence for the smoke test)
try:
    subprocess.Popen(["nvidia-smi","--query-gpu=index,utilization.gpu,utilization.memory,memory.used,power.draw",
                      "--format=csv,noheader","-l","10"],
                     stdout=open("/tmp/gpus.log","w"), stderr=subprocess.STDOUT)
except Exception as e:
    print("gpu monitor not started:",e)
ready=False
for i in range(72):
    try:
        import urllib.request as ur
        ur.urlopen("http://127.0.0.1:8188/history", timeout=3)
        ready=True; break
    except Exception: time.sleep(5)
if not ready:
    raise RuntimeError("ComfyUI did not listen in time")
print("ComfyUI local ready")


In [ ]:

import json,time,urllib.request,urllib.error,os
BASE="http://127.0.0.1:8188"
wf=json.load(open("/tmp/h3_t2v.json"))

# ---- apply ACTIVE settings (instance widgets are consumed during expansion) ----
SUBID=(wf.get("definitions") or {}).get("subgraphs")[0]["id"]
inst=[n for n in wf["nodes"] if n["type"]==SUBID][0]
iw=inst.setdefault("widgets_values_named",{})
iw.update({
    "prompt":ACTIVE["prompt"], "value_1":float(ACTIVE["duration"]),
    "noise_seed":int(ACTIVE["seed"]),
    "unet_name":ACTIVE["unet"], "clip_name":ACTIVE["clip"],
    "vae_name":ACTIVE["vae"], "vae_name_1":ACTIVE["audio_vae"],
    "value":bool(ACTIVE["turbo"]), "lora_name":ACTIVE["lora"],
    "strength_model_1":float(ACTIVE["turbo_strength"]),
    "value_2":int(ACTIVE["turbo_steps"]),
})
for n in wf["nodes"]:
    if n["type"]=="ResolutionSelector":
        rw=n.setdefault("widgets_values_named",{})
        rw["aspect_ratio"]=ACTIVE["aspect"]; rw["megapixels"]=ACTIVE["megapixels"]
print("settings applied:",ACTIVE_PRESET,
      "| turbo",ACTIVE["turbo"],
      "| steps",(ACTIVE["turbo_steps"] if ACTIVE["turbo"] else 20),
      "| megapixels",ACTIVE["megapixels"],"(",ACTIVE["aspect"],")")

info=json.loads(urllib.request.urlopen(BASE+"/object_info").read())
MISSING=object()

def schema(ct):
    if ct not in info:
        raise RuntimeError("node type not registered: "+ct)
    s=info[ct]["input"]
    return set(s.get("required",{}))|set(s.get("optional",{}))

subs={s["id"]:s for s in ((wf.get("definitions") or {}).get("subgraphs") or [])}
main_links={l[0]:l for l in (wf.get("links") or [])}
instance_out={}
prompt={}
_next=[1000]
def alloc():
    _next[0]+=1
    return str(_next[0])

def main_value(link_id):
    l=main_links[link_id]
    src,slot=l[1],l[2]
    if str(src) in instance_out:
        return instance_out[str(src)](slot)
    return [str(src),slot]

def expand_instance(node):
    sub=subs[node["type"]]
    ilinks={l["id"]:l for l in (sub.get("links") or [])}
    inst_inputs={i["name"]:i for i in (node.get("inputs") or [])}
    wnamed=node.get("widgets_values_named") or {}

    def sub_input(slot):
        name=sub["inputs"][slot]["name"]
        ii=inst_inputs.get(name)
        if ii is not None and ii.get("link") is not None:
            return main_value(ii["link"])
        if name in wnamed:
            return wnamed[name]
        return MISSING

    idmap={}
    for n in sub["nodes"]:
        schema(n["type"])
        idmap[str(n["id"])]=alloc()

    def out_value(slot):
        out=(sub.get("outputs") or [])[slot]
        lk=ilinks[out["linkIds"][0]]
        return [idmap[str(lk["origin_id"])],lk["origin_slot"]]

    for n in sub["nodes"]:
        valid=schema(n["type"])
        linked=set(); inputs={}
        for inp in (n.get("inputs") or []):
            nm=inp["name"]
            if inp.get("link") is None: continue   # links always included: autogrow slots
            linked.add(nm)                          # (values.a) only exist after finalization
            lk=ilinks.get(inp["link"])
            if lk is None: raise RuntimeError("inner link missing: %s"%inp["link"])
            if lk["origin_id"]==-10:
                v=sub_input(lk["origin_slot"])
            else:
                oid=idmap.get(str(lk["origin_id"]))
                if oid is None: raise RuntimeError("inner origin missing: %s"%lk["origin_id"])
                v=[oid,lk["origin_slot"]]
            if v is MISSING: continue
            inputs[nm]=v
        named=n.get("widgets_values_named") or {}
        for nm in valid:
            if nm in inputs or nm in linked: continue
            if nm in named: inputs[nm]=named[nm]
        prompt[idmap[str(n["id"])]]={"class_type":n["type"],"inputs":inputs}
    return out_value

# pass 1: flatten every subgraph instance
for n in wf.get("nodes",[]):
    if n["type"] in subs:
        instance_out[str(n["id"])]=expand_instance(n)
        print("expanded subgraph",n["type"],"-> nodes:",len(prompt))

# pass 2: regular (UI graph) nodes
for n in wf.get("nodes",[]):
    t=n["type"]
    if t in subs: continue
    if t not in info:
        print("skipping UI-only node:",t,"#%s"%n["id"]); continue
    valid=schema(t)
    linked=set(); inputs={}
    for inp in (n.get("inputs") or []):
        nm=inp["name"]
        if inp.get("link") is None: continue
        linked.add(nm)
        v=main_value(inp["link"])
        if v is MISSING: continue
        inputs[nm]=v
    named=n.get("widgets_values_named") or {}
    for nm in valid:
        if nm in inputs or nm in linked: continue
        if nm in named: inputs[nm]=named[nm]
    prompt[str(n["id"])]={"class_type":t,"inputs":inputs}

print("flattened prompt nodes:",len(prompt))

# turbo sanity: one boolean must drive both switches (LoRA branch + step count)
if ACTIVE["turbo"]:
    pb=[nid for nid,n in prompt.items()
        if n["class_type"]=="PrimitiveBoolean" and n["inputs"].get("value") is True]
    if not pb: raise RuntimeError("turbo enabled but no PrimitiveBoolean(value=True) in prompt")
    print("turbo path armed: PrimitiveBoolean",pb,"-> steps",ACTIVE["turbo_steps"],"+ LoRA")
else:
    print("dense path: 20 steps, no LoRA")

# insert H3MultiStream after the model switch, feeding BasicGuider + BasicScheduler
schema("H3MultiStream")
targets=[]
for nid,node in prompt.items():
    if node["class_type"] in ("BasicGuider","BasicScheduler") and isinstance(node["inputs"].get("model"),list):
        targets.append((nid,node["inputs"]["model"]))
if not targets:
    raise RuntimeError("no BasicGuider/BasicScheduler with a linked model input")
src=targets[0][1]
h3_in={"model":src,"enabled":True,"second_gpu":-1,"exchange":"host","exchange_chunks":8,
       "sparse_attention":False,"dynamic_vram":"keep","vram_block_cache":False,
       "vram_reserve_gb":2.0,"sparse_vsa":False,"weight_cache":False,"cache_ram_reserve_gb":0.0}
valid=schema("H3MultiStream")
missing=[k for k in valid if k not in h3_in and k!="gpus" and not isinstance(h3_in.get(k),list)]
h3_in={k:v for k,v in h3_in.items() if k in valid}
prompt["9999"]={"class_type":"H3MultiStream","inputs":h3_in}
n_p=0
for nid,v in targets:
    if v==src:
        prompt[nid]["inputs"]["model"]=["9999",0]; n_p+=1
print("H3MultiStream inserted as node 9999, feeding",n_p,"consumers; model source",src)
if missing: print("WARNING: H3 inputs not supplied:",missing)

body={"prompt":prompt,"extra_data":{"preview_method":"latent2rgb"}}
try:
    r=urllib.request.Request(BASE+"/prompt",data=json.dumps(body).encode(),headers={"Content-Type":"application/json"})
    resp=urllib.request.urlopen(r,timeout=60).read().decode()
    print("submitted:",resp)
except urllib.error.HTTPError as ee:
    print("prompt failed",ee,"body:",ee.read().decode()[:6000])
    raise


In [ ]:

import json,time,urllib.request,os
pid_j=json.loads(resp)
pid=pid_j.get("prompt_id") or pid_j.get("id") or pid_j.get("prompt")
print("prompt_id",pid)
got=None
seen=set()
KEY=("[GPUs]","[MultiStream]","[VAE split]","[Cache]","UNSPLIT","unsplit on 1 GPU")
for i in range(1080):                      # 3 h: fast smoke = 4 steps @ 0.4 MP (~40-60 min est.)
    try:
        h=json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}",timeout=5).read())
        if h and h.get(pid):
            got=h.get(pid); break
    except Exception: pass
    if i%12==0:
        try:
            lines=[l for l in open("/tmp/comfy.log",errors="ignore").read().splitlines() if l.strip()]
            print(f"[{i*10}s]", lines[-1][:200] if lines else "(no log yet)")
            diag=[l for l in lines if any(k in l for k in KEY)]
            new=[l for l in diag if l not in seen]
            for l in new: print("   [h3ms]", l[:250])       # [GPUs] plan, active ranks, step times
            seen.update(new)
            if any(("UNSPLIT" in l or "unsplit on 1 GPU" in l) for l in new):
                raise RuntimeError("single-GPU fallback detected: "+new[-1][:250])
        except RuntimeError: raise
        except Exception: pass
        try:   # live per-GPU util: index, util.gpu, util.mem, mem.used, power
            g=[l for l in open("/tmp/gpus.log",errors="ignore").read().splitlines() if l.strip()][-2:]
            for gl in g: print("   [gpu]", gl[:160])
        except Exception: pass
    time.sleep(10)
if not got: raise RuntimeError("smoke did not finish within 3 h")
# ---- dual-GPU verification: both T4s must actually be in the split ----
cl=open("/tmp/comfy.log",errors="ignore").read()
ev=[l for l in cl.splitlines() if ("[GPUs]" in l or "[MultiStream] active" in l or "UNSPLIT" in l or "unsplit on 1 GPU" in l)]
for l in ev: print("h3ms:",l[:300])
if "UNSPLIT" in cl or "running unsplit on 1 GPU" in cl:
    raise RuntimeError("single-GPU fallback detected -- smoke did NOT use both T4s")
if not any("active: 2 ranks" in l for l in ev):
    raise RuntimeError("no '[MultiStream] active: 2 ranks' evidence in comfy.log -- cannot confirm dual-GPU")
print("DUAL-GPU CONFIRMED: active: 2 ranks")
try:
    lastg=[l for l in open("/tmp/gpus.log",errors="ignore").read().splitlines() if l.strip()][-1:]
    print("last gpu sample:", lastg[0][:160] if lastg else "(none)")
except Exception: pass
st=got.get("status",{})
print("smoke status:",st.get("status_str"),"completed:",st.get("completed"),"/",st.get("remaining"))
if st.get("status_str")=="error": raise RuntimeError("smoke run had execution errors: "+json.dumps(st)[:2000])
print("smoke finished; outputs:")
for root,_,files in os.walk("/tmp/ComfyUI/output"):
    for f in files:
        if f.lower().endswith((".mp4",".webm",".mov",".webp",".gif",".png")):
            print("found:",os.path.join(root,f))


In [ ]:

import subprocess,time,re,urllib.request,json,os
from pathlib import Path
Path("/tmp/h3_api").mkdir(parents=True,exist_ok=True)
ok=False
for u in ["https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
          "https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64"]:
    try:
        data=urllib.request.urlopen(u,timeout=120).read()
        if len(data)>1000000:
            open("/tmp/h3_api/cloudflared","wb").write(data); ok=True; break
        print("cf dl too small from",u,len(data))
    except Exception as e:
        print("cf dl failed:",u,e)
if not ok: raise RuntimeError("cloudflared download failed")
os.chmod("/tmp/h3_api/cloudflared",0o755)
log=open("/tmp/h3_api/cf.log","w")
proc=subprocess.Popen(["/tmp/h3_api/cloudflared","tunnel","--url","http://127.0.0.1:8188"],stdout=log,stderr=subprocess.STDOUT)
url=None; pat=re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for i in range(90):
    t=open("/tmp/h3_api/cf.log","r",encoding="utf-8",errors="ignore").read()
    m=pat.search(t)
    if m:
        try:
            r=urllib.request.urlopen(m.group(0)+"/history",timeout=5)
            if r.status==200: url=m.group(0); break
        except Exception: pass
    time.sleep(2)
if not url: raise RuntimeError("cloudflare tunnel not healthy")
print("Verified public endpoint:",url)
print("\n================ MINIMAX H3 — GENERATION READY ================")
print("Base URL:"); print("  "+url)
print("API prompt endpoint:"); print("  "+url+"/prompt")
print("Example:")
print("===============================================================")
